# Pooled SpherIDiff: Pre-training and Regional Fine-Tuning

This notebook tests a hierarchical form of SpherIDiff, in which the blocks run at several resolution levels, as the levels of a U-Net. Pooling between levels is spectral: going down, the field is analysed at the band limit of the finer level, the degrees below the band limit of the coarser level are kept, and the result is synthesised on the coarser grid. Going up, the coarse field is synthesised on the finer grid and added to the skip connection of the same level. The width doubles at each level down. Every level is an equiangular grid with poles, and its band limit is the largest degree for which the least-squares fit has full rank, min(H - 1, W/2). The spectral stages and the pooling run in 16-bit precision.

The experiment crosses two choices, the number of levels and the form of the spectral weight:

| Variant | Levels and band limits | Widths | Spectral weight | Parameters |
|---|---|---|---|---|
| `pooled3_dense` | 3: L = 31, 15, 7 | 64, 128, 256 | dense K x K | 2,658,238 |
| `pooled3_rank32` | 3: L = 31, 15, 7 | 64, 128, 256 | D + P Q^T, r = 32 | 864,910 |
| `pooled4_dense` | 4: L = 31, 15, 7, 3 | 32, 64, 128, 256 | dense K x K | 2,674,448 |
| `pooled4_rank32` | 4: L = 31, 15, 7, 3 | 32, 64, 128, 256 | D + P Q^T, r = 32 | 882,408 |

Each variant is pre-trained on the global ERA5 data at 5.6 degrees for 25,000 steps (`pilot_25k`) and evaluated under the fixed protocol `final_v1` on the test year 2021. It is then fine-tuned for 5,000 steps (`finetune`) on the regional patch at 1.5 degrees over South Asia, and evaluated under the same protocol on that patch, once directly after the transfer of the weights (zero-shot) and once after fine-tuning.

On the patch, every level uses the Slepian basis of the patch at its own band limit: L' = 120 at the finest level, the least-squares limit of the 1.5 degree grid, and half of it at each coarser level (60, 30, 15). The numbers of Slepian modes are 175, 44, 11 and 3. The pre-trained weight of each block is restricted to the modes of its level and kept frozen, and a trainable update of the same size is added, which starts at zero. The pointwise weights are fine-tuned as well. Pooling on the patch is a least-squares projection onto the Slepian functions of the coarser level, because Slepian bases at different band limits are not nested.

The notebook is designed for a background run on two T4 GPUs. It reads the global data cache from the dataset `thesyeddaniyal/era5-pilot-6degrees` and the regional data cache from the dataset `thesyeddaniyal/era5-1-5-southasia`. The pre-trained weights are created in this session and read from its own runs folder. The results are packed into `pooled_results.zip` in the working folder.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
RUNS_ROOT = WORK_ROOT / "runs"
FINAL_PROTOCOL = "final_v1"
ZERO_SHOT_SUFFIX = "_zeroshot"
FIRST_GPU_VARIANTS = ["pooled3_dense", "pooled4_dense"]
SECOND_GPU_VARIANTS = ["pooled3_rank32", "pooled4_rank32"]
VARIANTS = FIRST_GPU_VARIANTS + SECOND_GPU_VARIANTS
PRETRAINING = ["pilot_" + name for name in VARIANTS]
FINETUNING = ["finetune_" + name for name in VARIANTS]
ZERO_SHOT = [name + ZERO_SHOT_SUFFIX for name in FINETUNING]
ARCHIVE = WORK_ROOT / "pooled_results.zip"

## Inputs and hardware

The repository code is cloned from GitHub (`https://github.com/syeddaniyalg/spheri-diff`) into the working folder, which requires internet access for the notebook. Within each attached dataset, the data cache is the folder that holds `stats.npz`. When the expected input path of a dataset does not exist, the whole of `/kaggle/input` is searched for a folder of the expected name.

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/syeddaniyalg/spheri-diff"
CODE_ROOT = WORK_ROOT / "spheri-diff"
GLOBAL_INPUT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-pilot-6degrees"
REGIONAL_INPUT = INPUT_ROOT / "datasets" / "thesyeddaniyal" / "era5-1-5-southasia"


def locate_cache(expected, folder_name):
    roots = [expected] if expected.exists() else [INPUT_ROOT]
    for root in roots:
        for current, _, files in os.walk(root):
            if "stats.npz" in files and (expected.exists() or Path(current).name == folder_name):
                return Path(current)
    raise FileNotFoundError(f"no data cache named {folder_name} was found under {roots[0]}")


if not CODE_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(CODE_ROOT)], check=True)
GLOBAL_CACHE = locate_cache(GLOBAL_INPUT, "era5_5p6deg")
REGIONAL_CACHE = locate_cache(REGIONAL_INPUT, "era5_1p5deg_south_asia")
sys.path.insert(0, str(CODE_ROOT / "src"))
print("code:", CODE_ROOT)
print("global data:", GLOBAL_CACHE)
print("regional data:", REGIONAL_CACHE)
for index in range(torch.cuda.device_count()):
    print("gpu", index, torch.cuda.get_device_name(index))

## Running the jobs

Each GPU runs one queue of separate processes. The first GPU takes the dense variants and the second GPU the variants with r = 32, each pairing a 3-level and a 4-level model, so that the two queues need about the same time. For each variant, the queue runs five jobs in order: pre-training, the global final evaluation, fine-tuning, the regional final evaluation of the zero-shot model and the regional final evaluation of the fine-tuned model. A variant is therefore complete before the next one starts. With one GPU, both queues share it.

Every job prints its lines into the notebook, prefixed with the elapsed time, the GPU and the job name, and the same lines are kept in `runs/_logs/parallel.log`. The jobs and the packing of the results share one cell with a `try` and a `finally`, so the figures, the status file, the summary tables and the archive are produced even when a job fails or the cell is interrupted. The archive is also refreshed every 30 minutes.

In [ ]:
from spheridiff.analysis.final_table import write_final_summary
from spheridiff.kaggle.finalise import finalise, guarded, refresh_archive
from spheridiff.kaggle.jobs import final_job, finetune_job, place_queues, train_job
from spheridiff.kaggle.runner import ParallelRunner, RunContext

GPUS = torch.cuda.device_count()
CONTEXT = RunContext(CODE_ROOT, RUNS_ROOT, RUNS_ROOT / "_logs", GLOBAL_CACHE)
REGIONAL_SUMMARY = RUNS_ROOT / "regional"


def variant_jobs(name):
    pretraining, finetuning = "pilot_" + name, "finetune_" + name
    return [
        train_job(pretraining, GLOBAL_CACHE),
        final_job(pretraining, FINAL_PROTOCOL, cache_root=GLOBAL_CACHE),
        finetune_job(finetuning, RUNS_ROOT, REGIONAL_CACHE),
        final_job(finetuning, FINAL_PROTOCOL, finetuning + ZERO_SHOT_SUFFIX, REGIONAL_CACHE),
        final_job(finetuning, FINAL_PROTOCOL, cache_root=REGIONAL_CACHE),
    ]


def queue(variants):
    return [job for name in variants for job in variant_jobs(name)]


def regional_summary():
    REGIONAL_SUMMARY.mkdir(parents=True, exist_ok=True)
    write_final_summary(REGIONAL_SUMMARY, [(RUNS_ROOT, ZERO_SHOT + FINETUNING)])


PLACEMENT = [(0, queue(FIRST_GPU_VARIANTS)), (1, queue(SECOND_GPU_VARIANTS))]


def refresh():
    return refresh_archive(RUNS_ROOT, ARCHIVE)


RUNNER = None
try:
    RUNNER = ParallelRunner(place_queues(PLACEMENT, GPUS), CONTEXT, refresh=refresh)
    RUNNER.run()
finally:
    OUTCOMES = RUNNER.outcomes() if RUNNER else []
    RUNS_ROOT.mkdir(parents=True, exist_ok=True)
    guarded(RUNS_ROOT, "regional_summary", regional_summary)
    finalise(RUNS_ROOT, ARCHIVE, OUTCOMES, PRETRAINING, final_names=PRETRAINING)
print(OUTCOMES)
print(ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} MB")

## Results

The pre-training summaries list the steps, training time, time per step, peak memory and final loss of every variant, and the figures show the validation error of the four variants during pre-training. The global table (`final_evaluation_summary.txt`) lists the four pre-trained variants under the fixed protocol on the global test year. The regional table (`regional/final_evaluation_summary.txt`) lists every variant on the patch, zero-shot (0 trained steps) and after fine-tuning. Both tables give the number of trainable parameters, the trained steps, the forecast error and CRPS, and for each task and method the normalised error on the unobserved entries, with the reference of the task as the last row.

In [ ]:
import json

for name in PRETRAINING + FINETUNING:
    summary = RUNS_ROOT / name / "summary.json"
    print(name, json.loads(summary.read_text()) if summary.exists() else "no summary, see runs/_logs")

for table in [RUNS_ROOT / "final_evaluation_summary.txt", REGIONAL_SUMMARY / "final_evaluation_summary.txt"]:
    if table.exists():
        print(table.read_text(encoding="utf-8"))

In [ ]:
from IPython.display import Image, display

for figure in ["evaluation_curves.png", "training_loss.png"]:
    path = RUNS_ROOT / figure
    if path.exists():
        display(Image(filename=str(path)))